<a href="https://colab.research.google.com/github/ZOUMKAI/MachineLearning/blob/main/0701_Colab_LINE_Bot_with_GEMINI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 7.4 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://landing-skied-babbling.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://landing-skied-babbling.ngrok-free.dev


True

In [ ]:
from google import genai

client = genai.Client(api_key=gemini_api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="什麼是AI AGENT？"
)
print(response.text)

AI AGENT（AI 代理）是一個比傳統AI模型（如單純的聊天機器人或圖像生成器）更具自主性、目標導向和行動能力的系統。你可以將它想像成一個**「具備思考能力，並且能主動採取行動來達成特定目標的數位工作者」**。

它不再僅僅是接收指令然後給出回應，而是能夠**理解複雜的目標，自主規劃步驟，並利用各種工具來執行任務**。

**核心構成要素與工作原理：**

AI Agent 的工作流程通常包含以下幾個關鍵環節：

1.  **感知 (Perception)：**
    *   能夠從環境中獲取資訊。這可以是文字輸入、圖像、資料庫內容、網頁資訊、API 回應等。
    *   **舉例：** 接收用戶「幫我規劃一次北海道的五天四夜自由行」的指令，並讀取當前日期和用戶偏好。

2.  **思考/規劃 (Thought/Planning)：**
    *   這是 AI Agent 的「大腦」，通常由大型語言模型（LLM）或其他AI模型擔任。
    *   它會分析感知到的資訊，理解目標，並將其分解為更小的、可執行的子任務。
    *   它會根據自身記憶、學習到的知識和推理能力，制定出一步步的行動計劃。
    *   **舉例：**
        *   將「規劃北海道自由行」分解為：選擇交通工具、預訂機票酒店、安排每日行程、查詢當地景點美食、預算估計等。
        *   規劃執行順序：先查機票價格，再定酒店，然後排行程。

3.  **行動 (Action)：**
    *   AI Agent 不僅會思考，還會「動手」。它會調用各種外部工具（Tools）來執行規劃好的任務。
    *   **工具可以是：**
        *   **網頁瀏覽器：** 搜尋資訊。
        *   **API 接口：** 預訂機票酒店、發送郵件、更新日曆、訪問資料庫。
        *   **程式碼解釋器：** 執行Python程式碼進行數據分析或處理。
        *   **文件操作：** 讀寫文件。
    *   **舉例：**
        *   使用航班查詢 API 查詢機票價格和班次。
        *   使用酒店預訂 API 查詢酒店空房和價格。
        *   使用地圖 API 查詢景點位置和交通路線。

In [ ]:
def stateless_query(payload):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=payload
    )
    return response.text


In [ ]:
result = stateless_query("簡介明新科技大學")
print(result)

**明新科技大學 (Ming Hsin University of Science and Technology, 簡稱明新科大)** 是一所位於臺灣新竹縣湖口鄉的科技大學。

**主要特色與簡介：**

1.  **歷史悠久與發展：** 學校創立於1966年，前身為「明新工業專科學校」，經過多年耕耘與發展，於2002年正式改制為科技大學。其深厚的技職教育背景，使其在實務教學方面擁有豐富經驗。

2.  **地理位置優勢：** 明新科大坐落於桃竹苗地區，毗鄰新竹科學園區，這使得學校在與高科技產業的鏈結上具有得天獨厚的優勢，方便進行產學合作，並為學生提供豐富的實習與就業機會。

3.  **辦學宗旨與特色：**
    *   **務實致用：** 學校秉持「務實致用」的辦學理念，課程設計強調理論與實務並重，旨在培養學生具備解決實際問題的能力。
    *   **專業技能與人文素養：** 除了專業技能的培養，明新科大也重視學生的人文素養與品德教育，期望培育出品學兼優的現代人才。
    *   **產學合作：** 學校積極推動與企業界的產學合作計畫，提供多元的實習機會，並鼓勵學生考取專業證照，以提升畢業生的職場競爭力。

4.  **學院與學術領域：** 明新科大目前設有工學院、管理學院、服務事業與資訊學院等，涵蓋工程、管理、設計、資訊、觀光餐旅等多個專業領域。

5.  **就業導向：** 明新科大畢業生在企業界普遍受到歡迎，擁有良好的就業表現，尤其在桃竹苗地區的產業中，明新科大的畢業生扮演著重要的角色。

總體而言，明新科技大學是一所深耕技職教育，與產業脈動緊密結合，致力於培養國家發展所需專業技術與管理人才的優質學府。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):#因為這裡有限定只能用A1 來做開頭
            prompt = text[3:]
            reply_text = stateless_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Uca511fbc529970b435eac4d5c87f9471","events":[{"type":"message","message":{"type":"text","id":"615029627358020136","quoteToken":"M9xdl8-enxw0xK0JDzAWLlJ3DKRXtL-_NNjrr6fSZhHqxQ1TS6MZ4VuhbtKJXOq_D1JbqN7xJR2RTNFncnjtYL5Ypn3V5ObF_Y1-UiztqZGUmFKfKUoTnAugV1VykkfXhyhB0MRnXZIwGaj6sRHNXw","markAsReadToken":"mw6bAcHZatJ9MhpY_5yfzBskiy4y0R_N2ElML-ZEMAqDfQBUy7hQuqDgVruj8T-ewCdZE8852w4Zwj3C6q7N4CpZFDq05Bz-T1kEeWb2DOT8wNz9U5zIG41qV3OKIojsdvpKwYLR_BOwJ2UAH8NvPwHFP8HkMbeMdmvzUK0PxGxp0hso3UBYBO26WS25ooqKGBk-iDxHR6ZlC5zyFc7Vgg","text":"AI 簡介明新科技大學"},"webhookEventId":"01KS6RSEE13C69W0F12FVH160C","deliveryContext":{"isRedelivery":false},"timestamp":1779417528742,"source":{"type":"user","userId":"U4c28faa6b4288acea2070a1906546976"},"replyToken":"52e8f3346ba74e4e96d3d7993f0da960","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 02:38:59] "POST / HTTP/1.1" 200 -
